Configurando ambiente de trabalho

In [0]:
%sql
use catalog stack_overgol;

In [0]:
from pyspark.sql.functions import (
    col, lower, upper, trim, when, regexp_extract, regexp_replace,
    length, concat, lit, split, explode, substring, coalesce,
    expr, initcap, ceil, unix_timestamp
)

1 - Clientes

In [0]:
print("Iniciando leitura da tabela bronze.clientes...")
df = spark.table("bronze.clientes")

print(f"Quantidade inicial de registros: {df.count()}")

print("Filtrando clientes com id_cliente nulo...")
df = df.filter(col("id_cliente").isNotNull())

print(f"Quantidade após filtro de id_cliente: {df.count()}")

print("Tratando coluna origem...")
df = df.withColumn(
    "origem",
    when(lower(trim(col("origem"))).isin("web"), "Web")
    .when(lower(trim(col("origem"))).isin("app"), "App")
    .when(lower(trim(col("origem"))).isin("indicação", "indicacao"), "Indicação")
    .otherwise("Outro")
)

print("Extraindo ramal do telefone...")
df = df.withColumn(
    "ramal",
    regexp_extract(col("telefone"), r"(?i)r\.?\s*(\d+)", 1)
)

df = df.withColumn(
    "ramal",
    when(col("ramal") == "", None).otherwise(col("ramal"))
)

print("Limpando telefone...")
df = df.withColumn(
    "telefone_limpo",
    trim(
        regexp_replace(
            regexp_replace(
                regexp_replace(col("telefone"), r"(?i)r\.?\s*\d+", ""),
                r"\D", ""
            ),
            r"^55(?=\d{10,11}$)", ""
        )
    )
)

print("Formatando telefone...")
df = df.withColumn(
    "telefone_formatado",
    when(length(col("telefone_limpo")) == 10,
        concat(
            lit("("), substring("telefone_limpo", 1, 2), lit(") "),
            substring("telefone_limpo", 3, 4), lit("-"),
            substring("telefone_limpo", 7, 4)
        )
    ).when(length(col("telefone_limpo")) == 11,
        concat(
            lit("("), substring("telefone_limpo", 1, 2), lit(") "),
            substring("telefone_limpo", 3, 5), lit("-"),
            substring("telefone_limpo", 8, 4)
        )
    ).otherwise(None)
)

print("Tratando emails...")
df = df.withColumn(
    "email_tratado",
    lower(trim(col("email")))
)

df = df.withColumn(
    "email_tratado",
    when(
        col("email_tratado").isNotNull() & (~col("email_tratado").contains("@")),
        regexp_replace(
            col("email_tratado"),
            r"(gmail|yahoo|hotmail|outlook|uol)",
            r"@\1"
        )
    ).otherwise(col("email_tratado"))
)

df = df.withColumn(
    "email_tratado",
    regexp_replace(col("email_tratado"), r"@{2,}", "@")
)

print("Padronizando nome e sobrenome...")
df = df.withColumn("nome", initcap(col("nome")))
df = df.withColumn("sobrenome", initcap(col("sobrenome")))

print("Tratando gênero...")
df = df.withColumn(
    "genero",
    when(lower(trim(col("genero"))).isin("m", "masculino", "male", "masc"), "Masculino")
    .when(lower(trim(col("genero"))).isin("f", "feminino", "female", "fem"), "Feminino")
    .otherwise("Não Informado")
)

print("Convertendo datas...")
df = df.withColumn(
    "data_nascimento",
    coalesce(
        expr("try_to_date(data_nascimento, 'yyyy-MM-dd')"),
        expr("try_to_date(data_nascimento, 'dd/MM/yyyy')"),
        expr("try_to_date(data_nascimento, 'yyyy/MM/dd')"),
        expr("try_to_date(data_nascimento, 'MM-dd-yyyy')")
    )
)

df = df.withColumn(
    "data_cadastro",
    coalesce(
        expr("try_to_date(data_cadastro, 'yyyy-MM-dd')"),
        expr("try_to_date(data_cadastro, 'dd/MM/yyyy')"),
        expr("try_to_date(data_cadastro, 'yyyy/MM/dd')"),
        expr("try_to_date(data_cadastro, 'MM-dd-yyyy')")
    )
)

print("Padronizando estado...")
df = df.withColumn("estado", initcap(trim(col("estado"))))
df = df.withColumn("cidade", initcap(trim(col("cidade"))))

print("Validando estados...")
estados_validos = [
    "Acre", "Alagoas", "Amapá", "Amazonas", "Bahia", "Ceará",
    "Distrito Federal", "Espírito Santo", "Goiás", "Maranhão",
    "Mato Grosso", "Mato Grosso Do Sul", "Minas Gerais", "Pará",
    "Paraíba", "Paraná", "Pernambuco", "Piauí", "Rio De Janeiro",
    "Rio Grande Do Norte", "Rio Grande Do Sul", "Rondônia",
    "Roraima", "Santa Catarina", "São Paulo", "Sergipe", "Tocantins"
]

df = df.withColumn(
    "estado_original",
    col("estado")
)

df = df.withColumn(
    "cidade_original",
    col("cidade")
)

df = df.withColumn(
    "estado",
    when(
        col("estado_original").isin(estados_validos),
        col("estado_original")
    ).otherwise(col("cidade_original"))
)

df = df.withColumn(
    "cidade",
    when(
        col("estado_original").isin(estados_validos),
        col("cidade_original")
    ).otherwise(col("estado_original"))
)

print("Removendo duplicados por id_cliente...")
antes = df.count()

df = df.dropDuplicates(["id_cliente"])

depois = df.count()

print(f"Duplicados removidos: {antes - depois}")

print("Selecionando colunas finais...")
df = df.select(
    col("id_cliente").cast("string"),
    col("nome").cast("string").alias("nome_cliente"),
    col("sobrenome").cast("string").alias("sobrenome_cliente"),
    col("email_tratado").alias("email_cliente"),
    col("telefone_formatado").alias("telefone_cliente"),
    col("ramal").alias("ramal_cliente"),
    col("genero").cast("string").alias("genero_cliente"),
    col("data_nascimento").alias("data_nascimento_cliente"),
    col("data_cadastro").alias("data_cadastro_cliente"),
    col("endereco").cast("string").alias("endereco_cliente"),
    col("cidade").cast("string").alias("cidade_cliente"),
    col("estado").cast("string").alias("estado_cliente"),
    col("pais").cast("string").alias("pais_cliente"),
    col("origem").alias("origem_cliente")
)

print("Gravando tabela silver.clientes...")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clientes")

print("Processamento concluído com sucesso!")
print(f"Total final de registros: {df.count()}")

Iniciando leitura da tabela bronze.clientes...
Quantidade inicial de registros: 61345
Filtrando clientes com id_cliente nulo...
Quantidade após filtro de id_cliente: 61345
Tratando coluna origem...
Extraindo ramal do telefone...
Limpando telefone...
Formatando telefone...
Tratando emails...
Padronizando nome e sobrenome...
Tratando gênero...
Convertendo datas...
Padronizando estado...
Validando estados...
Removendo duplicados por id_cliente...
Duplicados removidos: 3023
Selecionando colunas finais...
Gravando tabela silver.clientes...
Processamento concluído com sucesso!
Total final de registros: 58322


In [0]:
%sql
SELECT
    COUNT(DISTINCT CASE 
        WHEN p.id_cliente IS NOT NULL 
         AND c.id_cliente IS NOT NULL 
        THEN p.id_cliente 
    END) AS id_cliente_ambas,

    COUNT(DISTINCT CASE 
        WHEN c.id_cliente IS NOT NULL 
         AND p.id_cliente IS NULL 
        THEN c.id_cliente 
    END) AS id_cliente_cliente,

    COUNT(DISTINCT CASE 
        WHEN p.id_cliente IS NOT NULL 
         AND c.id_cliente IS NULL 
        THEN p.id_cliente 
    END) AS id_cliente_pedidos

FROM silver.clientes c
FULL OUTER JOIN silver.pedidos p
    ON c.id_cliente = p.id_cliente;

id_cliente_ambas,id_cliente_cliente,id_cliente_pedidos
50234,8088,0


2 - Dispositivos por cliente

In [0]:
print("Iniciando leitura da tabela bronze.clientes...")
df = spark.table("bronze.clientes")

print(f"Quantidade inicial de registros: {df.count()}")

print("Explodindo device_ids...")
df_dispositivo = df.select(
    col("id_cliente").cast("string"),
    explode(
        split(col("device_ids"), ";")
    ).alias("id_dispositivo")
)

print("Removendo espaços em branco dos dispositivos...")
df_dispositivo = df_dispositivo.withColumn(
    "id_dispositivo",
    trim(col("id_dispositivo"))
)

print("Filtrando dispositivos vazios...")
antes = df_dispositivo.count()

df_dispositivo = df_dispositivo.filter(col("id_dispositivo") != "")

depois = df_dispositivo.count()

print(f"Registros vazios removidos: {antes - depois}")

print("Removendo duplicados...")
antes = df_dispositivo.count()

df_dispositivo = df_dispositivo.dropDuplicates(["id_cliente", "id_dispositivo"])

depois = df_dispositivo.count()

print(f"Duplicados removidos: {antes - depois}")

print("Gravando tabela silver.clientes_dispositivo...")

df_dispositivo.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clientes_dispositivo")

print("Processamento concluído com sucesso!")
print(f"Total final de registros: {df.count()}")

Iniciando leitura da tabela bronze.clientes...
Quantidade inicial de registros: 61345
Explodindo device_ids...
Removendo espaços em branco dos dispositivos...
Filtrando dispositivos vazios...
Registros vazios removidos: 0
Removendo duplicados...
Duplicados removidos: 3015
Gravando tabela silver.clientes_dispositivo...
Processamento concluído com sucesso!
Total final de registros: 61345


3 - Catálogos Produto

In [0]:
print("Iniciando leitura da tabela bronze.catalogo_produtos...")
df = spark.table("bronze.catalogo_produtos")

print(f"Quantidade inicial de registros: {df.count()}")

print("Filtrando produtos com id_produto nulo...")
df = df.filter(col("id_produto").isNotNull())

print(f"Quantidade após filtro de id_produto: {df.count()}")

print("Tratando coluna ativo...")
df = df.withColumn(
    "ativo",
    when(
        lower(trim(col("ativo"))).isin("s", "sim", "yes", "1"),
        True
    ).when(
        lower(trim(col("ativo"))).isin("n", "nao", "não", "no", "0"),
        False
    ).otherwise(None)
)

print("Tratando categoria...")
df = df.withColumn(
    "categoria_tratada",
    lower(trim(col("categoria")))
)

df = df.withColumn(
    "categoria_tratada",
    regexp_replace(col("categoria_tratada"), r"[0-9@]", "")
)

df = df.withColumn(
    "categoria_tratada",
    when(col("categoria_tratada").isin(
        "eletronicos", "eletronico", "elet", "electronico", "electronics"
    ), "Eletrônicos")

    .when(col("categoria_tratada").isin(
        "vestuario", "vestu", "vest", "vestuarios", "moda", "roupa", "roupas"
    ), "Vestuário")

    .when(col("categoria_tratada").isin(
        "casa", "cas", "casa e jardim", "lar"
    ), "Casa")

    .when(col("categoria_tratada").isin(
        "esportes", "esporte", "esport", "esp", "sport", "sports"
    ), "Esportes")

    .when(col("categoria_tratada").isin(
        "beleza", "bel", "belz", "cosmeticos", "cosméticos"
    ), "Beleza")

    .when(col("categoria_tratada").isin(
        "automotivo", "autom", "aut", "automotiv", "auto"
    ), "Automotivo")

    .when(col("categoria_tratada").isin(
        "brinquedo", "brinquedos", "brin", "brinq", "toys"
    ), "Brinquedos")

    .when(col("categoria_tratada").isin(
        "moveis", "mov", "mveis", "móveis", "furniture"
    ), "Móveis")

    .when(lower(col("nome_produto")).rlike(
        "shampoo automotivo|pastilha de freio|oleo de motor|óleo de motor"
    ), "Automotivo")

    .when(lower(col("nome_produto")).rlike(
        "camiseta|bermuda|chinelo|roupão|roupao"
    ), "Vestuário")

    .when(lower(col("nome_produto")).rlike(
        "mesa de canto|mesa de centro"
    ), "Móveis")

    .when(lower(col("nome_produto")).rlike(
        "halteres|bloco de yoga|patins"
    ), "Esportes")

    .when(lower(col("nome_produto")).rlike(
        "creme para as mãos|creme para as maos|óleo corporal|oleo corporal|delineador|modelador de cachos|sabonete líquido|sabonete liquido"
    ), "Beleza")

    .when(lower(col("nome_produto")).rlike(
        "smartphone|impressora|controle sem fio|cabo usb|suporte para notebook"
    ), "Eletrônicos")

    .when(lower(col("nome_produto")).rlike(
        "bamboleo"
    ), "Brinquedos")

    .when(lower(col("nome_produto")).rlike(
        "micro-ondas|microondas|multiprocessador|aspirador|tábua de corte|tabua de corte|panelas"
    ), "Casa")
)

print("Tratando peso dos produtos...")
df = df.withColumn(
    "peso_kg",
    when(lower(trim(col("peso_kg"))) == "null", None)
    .otherwise(col("peso_kg").cast("double"))
)

print("Tratando estoque disponível...")
df = df.withColumn(
    "estoque_disponivel",
    when(
        lower(trim(col("estoque_disponivel"))) == "null",
        0
    ).otherwise(col("estoque_disponivel").cast("int"))
)

print("Tratando preços...")
df = df.withColumn(
    "preco_tratado",
    when(
        lower(trim(col("preco"))) == "null",
        None
    ).otherwise(col("preco"))
)

df = df.withColumn(
    "preco_tratado",
    regexp_replace(col("preco_tratado"), r"[R$\s]", "")
)

df = df.withColumn(
    "preco_tratado",
    regexp_replace(col("preco_tratado"), ",", ".")
)

df = df.withColumn(
    "preco_tratado",
    col("preco_tratado").cast("double")
)

df = df.withColumn(
    "preco_tratado",
    when(col("preco_tratado") <= 0, None)
    .otherwise(col("preco_tratado"))
)

print("Convertendo datas...")
df = df.withColumn(
    "data_cadastro_produto",
    coalesce(
        expr("try_to_date(data_cadastro_produto, 'yyyy-MM-dd')"),
        expr("try_to_date(data_cadastro_produto, 'dd/MM/yyyy')"),
        expr("try_to_date(data_cadastro_produto, 'yyyy/MM/dd')"),
        expr("try_to_date(data_cadastro_produto, 'MM-dd-yyyy')")
    )
)

print("Removendo duplicados...")
antes = df.count()

df = df.dropDuplicates(["id_produto"])

depois = df.count()

print(f"Duplicados removidos: {antes - depois}")

print("Selecionando colunas finais...")
df = df.select(
    col("id_produto").cast("string"),
    col("nome_produto").cast("string"),
    col("categoria_tratada").cast("string").alias("categoria_produto"),
    col("preco_tratado").alias("preco_produto"),
    col("fornecedor").cast("string").alias("fornecedor_produto"),
    col("peso_kg").alias("peso_kg_produto"),
    col("estoque_disponivel").alias("estoque_produto"),
    col("ativo").cast("boolean").alias("produto_ativo"),
    col("data_cadastro_produto")
)

print("Gravando tabela silver.catalogo_produtos...")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.catalogo_produtos")

print("Processamento concluído com sucesso!")
print(f"Total final de registros: {df.count()}")

Iniciando leitura da tabela bronze.catalogo_produtos...
Quantidade inicial de registros: 517
Filtrando produtos com id_produto nulo...
Quantidade após filtro de id_produto: 517
Tratando coluna ativo...
Tratando categoria...
Tratando peso dos produtos...
Tratando estoque disponível...
Tratando preços...
Convertendo datas...
Removendo duplicados...
Duplicados removidos: 0
Selecionando colunas finais...
Gravando tabela silver.catalogo_produtos...
Processamento concluído com sucesso!
Total final de registros: 517


4 - Trilha de ações do usuário

In [0]:
print("Iniciando leitura da tabela bronze.clickstream...")
df = spark.table("bronze.clickstream")

print(f"Quantidade inicial de registros: {df.count()}")

print("Filtrando registros com id_evento ou id_cliente nulos...")
df = df.filter(col("id_evento").isNotNull() & col("id_cliente").isNotNull())

print(f"Quantidade após filtros obrigatórios: {df.count()}")

print("Tratando tipo de evento...")
df = df.withColumn(
    "tipo_normalizado",
    lower(trim(col("tipo_evento")))
)

df = df.withColumn(
    "tipo_normalizado",
    regexp_replace(col("tipo_normalizado"), r"[-\s]", "_")
)

df = df.withColumn(
    "tipo_padronizado",
    when(col("tipo_normalizado").isin(
        "pageview", "page_view", "pv"
    ), "Vizualização de Página")

    .when(col("tipo_normalizado").isin(
        "add_to_cart", "addtocart", "adicionar", "add_cart"
    ), "Adicionar no Carrinho")

    .when(col("tipo_normalizado").isin(
        "login", "log_in", "signin", "sign_in"
    ), "Login")

    .when(col("tipo_normalizado").isin(
        "purchase", "compra", "buy", "bought"
    ), "Compra")

    .when(col("tipo_normalizado").isin(
        "checkout", "check_out", "pagamento"
    ), "Pagamento")

    .when(col("tipo_normalizado").isin(
        "search", "srch", "busca", "pesquisa"
    ), "Busca")

    .when(col("tipo_normalizado").isin(
        "abandon_cart", "abandono", "abandon", "cart_abandon"
    ), "Abandono de Carrinho")

    .otherwise("Outros")
)

print("Tratando canal...")
df = df.withColumn(
    "canal_normalizado",
    lower(trim(col("canal")))
)

df = df.withColumn(
    "canal_normalizado",
    regexp_replace(col("canal_normalizado"), r"[\s\-]", "_")
)

df = df.withColumn(
    "canal_padronizado",
    when(col("canal_normalizado").isin(
        "web", "mobile_web"
    ), "Web")

    .when(col("canal_normalizado").isin(
        "app", "aplicativo"
    ), "App")

    .otherwise("Outros")
)

print("Tratando dispositivo...")
df = df.withColumn(
    "dispositivo_normalizado",
    lower(trim(col("dispositivo")))
)

df = df.withColumn(
    "dispositivo_padronizado",
    when(col("dispositivo_normalizado").isin(
        "desktop", "computador", "pc"
    ), "Desktop")

    .when(col("dispositivo_normalizado").isin(
        "mobile", "celular", "mob", "smartphone"
    ), "Mobile")

    .when(col("dispositivo_normalizado").isin(
        "tablet", "tab", "ipad"
    ), "Tablet")

    .otherwise("Outros")
)

print("Tratando origem da sessão...")
df = df.withColumn(
    "origem_sessao_tratada",
    lower(trim(col("origem_sessao")))
)

df = df.withColumn(
    "origem_sessao_tratada",
    when(col("origem_sessao_tratada") == "social", "Social")
    .when(col("origem_sessao_tratada") == "organic", "Orgânico")
    .when(col("origem_sessao_tratada").isin("paid_search", "paid"), "Busca Paga")
    .when(col("origem_sessao_tratada") == "direct", "Direto")
    .when(col("origem_sessao_tratada") == "email", "Email")
    .otherwise("Outros")
)

print("Tratando tempo de página...")
df = df.withColumn(
    "tempo_pagina_seg",
    when(col("tempo_pagina_seg") < 0, None)
    .otherwise(col("tempo_pagina_seg"))
)

print("Convertendo data_evento para timestamp...")
df = df.withColumn(
    "data_evento",
    coalesce(
        col("data_evento").cast("timestamp"),
        expr("try_cast(data_evento as timestamp)")
    )
)

print("Removendo duplicados...")
antes = df.count()

df = df.dropDuplicates(["id_evento"])

depois = df.count()

print(f"Duplicados removidos: {antes - depois}")

print("Selecionando colunas finais...")
df = df.select(
    col("id_evento").cast("string"),
    col("id_sessao").cast("string"),
    col("id_cliente").cast("string"),
    col("id_dispositivo").cast("string"),
    col("id_produto").cast("string"),
    col("tipo_padronizado").cast("string").alias("tipo_evento"),
    col("canal_padronizado").cast("string").alias("canal_evento"),
    col("dispositivo_padronizado").cast("string").alias("dispositivo_evento"),
    col("origem_sessao_tratada").cast("string").alias("origem_sessao"),
    col("data_evento"),
    col("tempo_pagina_seg").cast("int")
)

print("Gravando tabela silver.clickstream...")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clickstream")

print("Processamento concluído com sucesso!")
print(f"Total final de registros: {df.count()}")

Iniciando leitura da tabela bronze.clickstream...
Quantidade inicial de registros: 500000
Filtrando registros com id_evento ou id_cliente nulos...
Quantidade após filtros obrigatórios: 349447
Tratando tipo de evento...
Tratando canal...
Tratando dispositivo...
Tratando origem da sessão...
Tratando tempo de página...
Convertendo data_evento para timestamp...
Removendo duplicados...
Duplicados removidos: 0
Selecionando colunas finais...
Gravando tabela silver.clickstream...
Processamento concluído com sucesso!
Total final de registros: 349447


5 - Pedidos

In [0]:
print("Iniciando leitura da tabela bronze.pedidos...")
df = spark.table("bronze.pedidos")

print(f"Quantidade inicial de registros: {df.count()}")

print("Filtrando registros com id_pedido ou id_cliente nulos...")
df = df.filter(col("id_pedido").isNotNull() & col("id_cliente").isNotNull())

print(f"Quantidade após filtros obrigatórios: {df.count()}")

print("Convertendo data_pedido...")
df = df.withColumn(
    "data_pedido",
    coalesce(
        expr("try_to_date(data_pedido, 'yyyy-MM-dd')"),
        expr("try_to_date(data_pedido, 'yyyy/MM/dd')"),
        expr("try_to_date(data_pedido, 'yyyy/dd/MM')"),
        expr("try_to_date(data_pedido, 'dd/MM/yyyy')"),
        expr("try_to_date(data_pedido, 'MM-dd-yyyy')")
    )
)

print("Tratando valor_pedido...")
df = df.withColumn(
    "valor_pedido",
    expr("""
        try_cast(
            regexp_replace(
                regexp_replace(trim(valor_pedido), 'R\\$\\s*', ''),
                ',', '.'
            ) as double
        )
    """)
)

df = df.withColumn(
    "valor_pedido",
    when(col("valor_pedido") <= 0, None)
    .otherwise(col("valor_pedido"))
)

print("Tratando método de pagamento...")
df = df.withColumn(
    "metodo_pagamento",
    lower(regexp_replace(col("metodo_pagamento"), "[^a-zA-Z0-9]", ""))
)

df = df.withColumn(
    "metodo_pagamento",
    when(col("metodo_pagamento").isin("pix", "p1x", "plx"), "Pix")
    .when(col("metodo_pagamento").isin("boleto", "b0leto", "bol", "blt"), "Boleto")
    .when(col("metodo_pagamento").isin(
        "cartao", "crtao", "credito", "debito", "credit", "debit",
        "cartaocredito", "cartaodebito", "crt", "card"
    ), "Cartão")
    .otherwise("Outros")
)

print("Tratando quantidade...")
df = df.withColumn(
    "quantidade",
    when(lower(trim(col("quantidade"))) == "um", "1")
    .when(lower(trim(col("quantidade"))) == "dois", "2")
    .when(lower(trim(col("quantidade"))) == "tres", "3")
    .when(lower(trim(col("quantidade"))) == "três", "3")
    .when(lower(trim(col("quantidade"))) == "quatro", "4")
    .when(lower(trim(col("quantidade"))) == "cinco", "5")
    .otherwise(col("quantidade"))
)

df = df.withColumn(
    "quantidade",
    expr("try_cast(quantidade as double)")
)

df = df.withColumn(
    "quantidade",
    when(col("quantidade") < 0, None)
    .otherwise(col("quantidade").cast("int"))
)

print("Tratando status do pedido...")
df = df.withColumn(
    "status_tratado",
    lower(trim(col("status")))
)

df = df.withColumn(
    "status_tratado",
    when(col("status_tratado").isin(
        "aprovado", "aprovadoo", "aprov", "apr", "approved"
    ), "Aprovado")
    .when(col("status_tratado").isin(
        "reembolsado", "reembolso", "reemb", "reembolsad", "reembolsadoo", "refunded"
    ), "Reembolsado")
    .when(col("status_tratado").isin(
        "recusado", "recus", "recusadoo", "rec", "declined", "refused"
    ), "Recusado")
    .when(col("status_tratado").isin(
        "process", "processando", "proc", "processing", "em processamento"
    ), "Processando")
    .otherwise("Outros")
)

print("Removendo duplicados...")
antes = df.count()

df = df.dropDuplicates(["id_pedido"])

depois = df.count()

print(f"Duplicados removidos: {antes - depois}")

print("Selecionando colunas finais...")
df = df.select(
    col("id_pedido").cast("string"),
    col("id_cliente").cast("string"),
    col("id_produto").cast("string"),
    col("valor_pedido"),
    col("data_pedido"),
    col("metodo_pagamento"),
    col("status_tratado").cast("string").alias("status_pedido"),
    col("quantidade").alias("quantidade_produto"),
    lit(None).cast("date").alias("data_prevista_entrega")
)

print("Gravando tabela silver.pedidos...")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.pedidos")

print("Processamento concluído com sucesso!")
print(f"Total final de registros: {df.count()}")

Iniciando leitura da tabela bronze.pedidos...
Quantidade inicial de registros: 314900
Filtrando registros com id_pedido ou id_cliente nulos...
Quantidade após filtros obrigatórios: 314900
Convertendo data_pedido...
Tratando valor_pedido...
Tratando método de pagamento...
Tratando quantidade...
Tratando status do pedido...
Removendo duplicados...
Duplicados removidos: 0
Selecionando colunas finais...
Gravando tabela silver.pedidos...
Processamento concluído com sucesso!
Total final de registros: 314900


6 - Avaliações

In [0]:
print("Iniciando leitura da tabela bronze.avaliacoes...")
df = spark.table("bronze.avaliacoes")

print(f"Quantidade inicial de registros: {df.count()}")

print("Filtrando registros com id_avaliacao nulo...")
df = df.filter(col("id_avaliacao").isNotNull())

print(f"Quantidade após filtro obrigatório: {df.count()}")

print("Tratando coluna recomenda...")
df = df.withColumn(
    "recomenda",
    when(
        lower(trim(col("recomenda"))).isin("s", "sim", "yes", "1", "true"),
        True
    ).when(
        lower(trim(col("recomenda"))).isin("n", "nao", "não", "no", "0", "false"),
        False
    ).otherwise(None)
)

print("Tratando nota_produto...")
df = df.withColumn(
    "nota_produto",
    when(lower(trim(col("nota_produto"))) == "ótimo", 5)
    .when(lower(trim(col("nota_produto"))) == "bom", 4)
    .when(lower(trim(col("nota_produto"))) == "regular", 3)
    .when(lower(trim(col("nota_produto"))) == "ruim", 2)
    .when(lower(trim(col("nota_produto"))) == "péssimo", 1)
    .otherwise(col("nota_produto").cast("int"))
)

df = df.withColumn(
    "nota_produto",
    when(col("nota_produto") < 1, 1)
    .when(col("nota_produto") > 5, 5)
    .otherwise(col("nota_produto"))
)

print("Tratando nota_nps...")
df = df.withColumn(
    "nota_nps",
    when(lower(trim(col("nota_nps"))) == "ótimo", 10)
    .when(lower(trim(col("nota_nps"))) == "bom", 8)
    .when(lower(trim(col("nota_nps"))) == "regular", 5)
    .when(lower(trim(col("nota_nps"))) == "ruim", 2)
    .when(lower(trim(col("nota_nps"))) == "péssimo", 0)
    .otherwise(col("nota_nps").cast("int"))
)

df = df.withColumn(
    "nota_nps",
    when(col("nota_nps") < 0, 0)
    .when(col("nota_nps") > 10, 10)
    .otherwise(col("nota_nps"))
)

print("Convertendo data_avaliacao...")
df = df.withColumn(
    "data_avaliacao",
    coalesce(
        expr("try_to_date(data_avaliacao, 'yyyy-MM-dd HH:mm:ss')"),
        expr("try_to_date(data_avaliacao, 'yyyy/dd/MM HH:mm:ss')"),
        expr("try_to_date(data_avaliacao, 'yyyy-MM-dd')"),
        expr("try_to_date(data_avaliacao, 'dd/MM/yyyy')"),
        expr("try_to_date(data_avaliacao, 'yyyy/MM/dd')")
    )
)

print("Carregando tabela silver.pedidos...")
pedidos = spark.table("silver.pedidos").select(
    col("id_pedido"),
    col("data_pedido")
)

print("Realizando join com pedidos...")
df = df.join(pedidos, on="id_pedido", how="left")

print("Preenchendo data_avaliacao com data_pedido quando necessário...")
df = df.withColumn(
    "data_avaliacao",
    coalesce(col("data_avaliacao"), col("data_pedido"))
)

df = df.drop("data_pedido")

print("Removendo duplicados...")
antes = df.count()

df = df.dropDuplicates(["id_avaliacao"])

depois = df.count()

print(f"Duplicados removidos: {antes - depois}")

print("Selecionando colunas finais...")
df = df.select(
    col("id_avaliacao").cast("string"),
    col("id_pedido").cast("string"),
    col("id_cliente").cast("string"),
    col("id_produto").cast("string"),
    col("nota_produto"),
    col("comentario").cast("string").alias("comentario_avaliacao"),
    col("nota_nps"),
    col("recomenda").alias("recomenda_produto"),
    col("data_avaliacao")
)

print("Gravando tabela silver.avaliacoes...")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.avaliacoes")

print("Processamento concluído com sucesso!")
print(f"Total final de registros: {df.count()}")

Iniciando leitura da tabela bronze.avaliacoes...
Quantidade inicial de registros: 156832
Filtrando registros com id_avaliacao nulo...
Quantidade após filtro obrigatório: 156832
Tratando coluna recomenda...
Tratando nota_produto...
Tratando nota_nps...
Convertendo data_avaliacao...
Carregando tabela silver.pedidos...
Realizando join com pedidos...
Preenchendo data_avaliacao com data_pedido quando necessário...
Removendo duplicados...
Duplicados removidos: 0
Selecionando colunas finais...
Gravando tabela silver.avaliacoes...
Processamento concluído com sucesso!
Total final de registros: 156832


7 - Tickets Suporte

In [0]:
print("Iniciando leitura da tabela bronze.suporte_tickets...")
df = spark.table("bronze.suporte_tickets")

print(f"Quantidade inicial de registros: {df.count()}")

print("Filtrando registros com ticket_id nulo...")
df = df.filter(col("ticket_id").isNotNull())

print(f"Quantidade após filtro obrigatório: {df.count()}")

print("Tratando tipo_problema...")
df = df.withColumn(
    "tipo_problema",
    when(
        lower(trim(col("tipo_problema"))).isin(
            "pro", "produto", "p3oduto", "prod", "product"
        ), "Produto"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "pag", "pagamento", "p4gamento", "pay", "payment"
        ), "Pagamento"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "entrega", "3ntrega", "entr", "ent", "delay", "del", "shipping"
        ), "Entrega"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "reembolso", "reemb", "r3embolso", "refund", "ref"
        ), "Reembolso"
    ).otherwise("Outro")
)

print("Convertendo datas...")
df = df.withColumn(
    "data_abertura",
    col("data_abertura").cast("timestamp")
)

df = df.withColumn(
    "data_resolucao",
    col("data_resolucao").cast("timestamp")
)

print("Calculando tempo_resolucao_horas...")
df = df.withColumn(
    "tempo_resolucao_horas",
    when(
        col("data_abertura").isNull() | col("data_resolucao").isNull(),
        None
    ).when(
        col("data_resolucao") < col("data_abertura"),
        None
    ).otherwise(
        ceil(
            (unix_timestamp(col("data_resolucao")) - unix_timestamp(col("data_abertura"))) / 3600
        ).cast("int")
    )
)

print("Tratando nota_avaliacao...")
df = df.withColumn(
    "nota_avaliacao",
    col("nota_avaliacao").cast("int")
)

df = df.withColumn(
    "nota_avaliacao",
    when(col("nota_avaliacao") < 1, None)
    .when(col("nota_avaliacao") > 5, None)
    .otherwise(col("nota_avaliacao"))
)

print("Tratando sentimento...")
if "sentimento" in df.columns:
    df = df.withColumn(
        "sentimento",
        when(lower(trim(col("sentimento"))).isin("positivo", "pos", "positive"), "Positivo")
        .when(lower(trim(col("sentimento"))).isin("negativo", "neg", "negative"), "Negativo")
        .when(lower(trim(col("sentimento"))).isin("neutro", "neu", "neutral"), "Neutro")
        .otherwise(None)
    )
else:
    print("Coluna sentimento não encontrada. Criando coluna nula...")
    df = df.withColumn("sentimento", lit(None).cast("string"))

print("Tratando status do ticket...")
if "status" in df.columns:
    df = df.withColumn(
        "status_ticket",
        when(lower(trim(col("status"))).isin("aberto", "open", "novo"), "Aberto")
        .when(lower(trim(col("status"))).isin("fechado", "closed", "resolvido", "resolved"), "Fechado")
        .when(lower(trim(col("status"))).isin("em andamento", "em_andamento", "in_progress"), "Em Andamento")
        .otherwise("Outro")
    )
else:
    print("Coluna status não encontrada. Criando coluna nula...")
    df = df.withColumn("status_ticket", lit(None).cast("string"))

print("Removendo duplicados...")
antes = df.count()

df = df.dropDuplicates(["ticket_id"])

depois = df.count()

print(f"Duplicados removidos: {antes - depois}")

print("Selecionando colunas finais...")
df = df.select(
    col("ticket_id").cast("string"),
    col("id_cliente").cast("string"),
    col("id_pedido").cast("string"),
    col("tipo_problema"),
    col("data_abertura"),
    col("data_resolucao"),
    col("tempo_resolucao_horas"),
    col("agente_suporte").cast("string"),
    col("nota_avaliacao").alias("nota_avaliacao_problema"),
    col("sentimento"),
    col("status_ticket")
)

print("Gravando tabela silver.suporte_tickets...")

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.suporte_tickets")

print("Processamento concluído com sucesso!")
print(f"Total final de registros: {df.count()}")

Iniciando leitura da tabela bronze.suporte_tickets...
Quantidade inicial de registros: 34697
Filtrando registros com ticket_id nulo...
Quantidade após filtro obrigatório: 34697
Tratando tipo_problema...
Convertendo datas...
Calculando tempo_resolucao_horas...
Tratando nota_avaliacao...
Tratando sentimento...
Coluna sentimento não encontrada. Criando coluna nula...
Tratando status do ticket...
Coluna status não encontrada. Criando coluna nula...
Removendo duplicados...
Duplicados removidos: 0
Selecionando colunas finais...
Gravando tabela silver.suporte_tickets...
Processamento concluído com sucesso!
Total final de registros: 34697
